In [25]:
library(XML)
library(tidyverse)
library(glue)

# ==============================================================================
# 1. GENERADORES ATÓMICOS DE SINTAXIS CLOZE (Mantienen su lógica intacta)
# ==============================================================================

cloze_num <- function(resp, error = 0.0001, peso = 1, digresp = 4, diger = digresp) {
  val_formated <- formatC(resp, digits = digresp, format = "f")
  err_formated <- formatC(error, digits = diger, format = "f")
  glue("{{{peso}:NM:={val_formated}:{err_formated}}}")
}

cloze_mch <- function(resp = 1, opciones, peso = 1) {
  if (resp < 1 | resp > length(opciones)) stop("Índice de respuesta fuera de rango.")
  opciones_formateadas <- map2_chr(opciones, seq_along(opciones), function(opc, idx) {
    if (idx != resp) paste0("~%0%", opc) else paste0("~=", opc)
  })
  glue("{{{peso}:MULTICHOICE:{paste0(opciones_formateadas, collapse = '')}}}")
}

# ==============================================================================
# 2. INICIALIZADOR DEL CONTENEDOR XML
# ==============================================================================
iniciar_quiz_xml <- function(nombre_categoria) {
  doc <- newXMLDoc()
  root <- newXMLNode("quiz", doc = doc)
  
  question_cat <- newXMLNode("question", attrs = list(type = "category"), parent = root)
  category <- newXMLNode("category", parent = question_cat)
  newXMLNode("text", nombre_categoria, parent = category)
  info <- newXMLNode("info", attrs = list(format = "html"), parent = question_cat)  
  newXMLNode("text", "Categoría generada automáticamente", parent = info)
  newXMLNode("idnumber", parent = question_cat)
  
  return(list(doc = doc, root = root))
}

# ==============================================================================
# 3. EL MOTOR GENÉRICO Y COMPILADOR DE PLANTILLAS
# ==============================================================================


crear_pregunta_cloze <- function(moodle_quiz, nombre_pregunta, plantilla_texto, plantilla_tabla = NULL, lista_respuestas) {
  
  # Usamos .open y .close para cambiar los delimitadores de las variables a [[ ]]
  if (!is.null(plantilla_tabla)) {
    tabla_renderizada <- glue_data(lista_respuestas, plantilla_tabla, .open = "[[", .close = "]]")
    lista_respuestas$TABLA_EVALUACION <- tabla_renderizada
  }
  
  texto_final_html <- glue_data(lista_respuestas, plantilla_texto, .open = "[[", .close = "]]")
  
  # Estructura del nodo Cloze en XML
  questionpr <- newXMLNode("question", attrs = list(type = "cloze"), parent = moodle_quiz$root)
  nombrepr <- newXMLNode("name", parent = questionpr)
  newXMLNode("text", nombre_pregunta, parent = nombrepr)
  
  questiontext <- newXMLNode("questiontext", attrs = list(format = "html"), parent = questionpr)
  newXMLNode("text", as.character(texto_final_html), parent = questiontext)
  
  newXMLNode("generalfeedback", attrs = list(format = "html"), parent = questionpr)
  newXMLNode("penalty", "0.3333333", parent = questionpr)
  newXMLNode("hidden", "0", parent = questionpr)
  newXMLNode("idnumber", parent = questionpr)
}

In [26]:
setwd("~/Documents/Cursos/Laboratorio")
#df <- read.csv("PastureYield.csv", dec=",")
library(readxl)
library(tidyverse) 
dfo <- read_excel("DATOS CACHAMAS.xlsx")
dfc=dfo|>mutate(LongEstandar=if_else(LongEstandar>100,
                LongEstandar/10,
                LongEstandar), estanque=factor(estanque))
names(df)
#summary(df$estanque)
df=dfc|>filter(estanque=='R03')|>mutate(estanque=droplevels(estanque))
mod1=lm(peso~LongEstandar+LongHorquilla+LongTotal,data=df)
mod2=lm(peso~LongHorquilla,data=df)
#mod2=step(mod1, direction = "both") 
summary(mod2) 
t1=anova(mod2,mod1)
t1
#write_csv(dfc,"DATOS CACHAMAS_Cor.xlsx")
levels(df$estanque) 

[1] "estanque"      "peso"          "LongEstandar"  "LongHorquilla"
[5] "LongTotal"


Call:
lm(formula = peso ~ LongHorquilla, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-38.978  -8.978  -3.450  12.145  36.843 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)    -507.30      28.35  -17.89 2.39e-13 ***
LongHorquilla    37.53       1.49   25.19 4.63e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 17.63 on 19 degrees of freedom
Multiple R-squared:  0.9709,	Adjusted R-squared:  0.9694 
F-statistic: 634.6 on 1 and 19 DF,  p-value: 4.625e-16


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,19,5906.072,NA,NA,NA,NA
2,17,3562.605,2,2343.467,5.591264,0.01361401


[1] "R03"

In [27]:
t1=anova(mod1,mod2)
opc_dplyr <- c("modelo con las 3 variables",
               "modelo con LH+LT",
               "modelo con LH+LE",
               "modelo con LE+LT",
               "modelo con LT",
               "modelo con LE",
               "modelo con LH")

respuestas_dinamicas <- list(
  cloze_df1     = cloze_num(t1[1,1], digresp = 0, error = 0),
  cloze_df2     = cloze_num(t1[2,1], digresp = 0, error = 0),
  cloze_rss1    = cloze_num(t1[1,2], digresp = 0, error = 0),
  cloze_rss2    = cloze_num(t1[2,2], digresp = 0, error = 0),
  cloze_df      = cloze_num(t1[2,3], digresp = 0, error = 0),
  cloze_sos     = cloze_num(t1[2,4]),
  cloze_F       = cloze_num(t1[2,5]),
  cloze_pv      = cloze_num(t1[2,6]),
  cloze_modelo  = cloze_mch(resp=6,opciones=opc_dplyr) 
)

plantilla_tabla<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0pky">Res.df</th>
    <th class="tg-0pky">RSS</th>
    <th class="tg-0pky">Df</th>
    <th class="tg-0pky">Sum of Sq</th>
    <th class="tg-0pky">F</th>
    <th class="tg-0pky">Pr(&gt;F)</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0pky">[[cloze_df1]]</td>
    <td class="tg-0pky">[[cloze_rss1]]</td>
    <td class="tg-0pky"></td>
    <td class="tg-0pky"></td>
    <td class="tg-0pky"></td>
    <td class="tg-0pky"></td>
  </tr>
  <tr>
    <td class="tg-0pky">[[cloze_df2]]</td>
    <td class="tg-0pky">[[cloze_rss2]]</td>
    <td class="tg-0pky">[[cloze_df]]</td>
    <td class="tg-0pky">[[cloze_sos]]</td>
    <td class="tg-0pky">[[cloze_F]]</td>
    <td class="tg-0pky">[[cloze_pv]]</td>
  </tr>
</tbody>
</table>'  

plantilla_enunciado <-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>. Se quiere  ajustar un modelo
que permita explicar el peso de los animales en función de las longitudes. 
Deseamos decidir cuál modelo usar entre el modelo con las tres longitudes como variables explicativas
\\(E(P_i)=\\beta_0+\\beta_1LH_i+\\beta_2LE_i+\\beta_3LT_i\\) (lineal) y el 
modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\) (que solo incluye la variable LH). Complete la siguiente tabla 
de análisis de varianza que contiene la prueba de la hipótesis: <br> \\(H_0:\\)
"El modelo con todas las variables no aporta un ajuste significativamente mejor que el modelo con solo LH",
contra la alternativa  \\(H_1:\\) "El modelo con todas las variables ajusta  significativamente mejor 
que el el modelo con solo LH" <br>, [[TABLA_EVALUACION]] de acuerdo con los resultados mostrados en la tabla anterior
decidimos usar el modelo [[cloze_modelo]] </p>'  


## respecto del parámetro beta_1

opc_dplyr2a <- c("igual a","distinto de", "mayor que", "menor que" )
opc_dplyr2b <- c("no contribuye a", "es indiferente para ", "contribuye a", "no es útil para")
res=summary(mod2)
coe=res[['coefficients']]

respuestas_dinamicas2 <- list(
  cloze_coef0     = cloze_num(coe[1,1]),
  cloze_coef1   = cloze_num(coe[2,1]),
  cloze_ste0    = cloze_num(coe[1,2]),
  cloze_ste1    = cloze_num(coe[2,2]),
  cloze_t0    = cloze_num(coe[1,3]),
  cloze_t1    = cloze_num(coe[2,3]),
  cloze_sigB0 = cloze_mch(resp=2,opciones=opc_dplyr2a),
  cloze_sigB1 = cloze_mch(resp=2,opciones=opc_dplyr2a), 
  close_conclu = cloze_mch(resp=3,opciones=opc_dplyr2b)
) 

plantilla_tabla2<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0lax"></th>
    <th class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">Estimate</span></th>
    <th class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">Std. Error</span></th>
    <th class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">t value</span></th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0lax">Intercept</td>
    <td class="tg-0lax">[[cloze_coef0]]</td>
    <td class="tg-0lax">[[cloze_ste0]]</td>
    <td class="tg-0lax">[[cloze_t0]]</td>
  </tr>
  <tr>
    <td class="tg-0lax">LH</td>
    <td class="tg-0lax">[[cloze_coef1]]</td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_ste1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_t1]]</span></td>
  </tr>
</tbody>
</table>'  

plantilla_enunciado2<-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>.
Se quiere  ajustar un modelo que permita explicar el <code>peso</code> de los animales en función de las longitudes. 
Ajuste el modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\)  
<br> Complete la siguiente tabla que contiene los parámetros estimados, sus errores
estándar estimados y el valor del estadístico  \\(t)\\ que prueba la hipótesis \\(\\beta_i=0)\\ 
contra \\(\\beta_i \\neq 0)\\  <br>[[TABLA_EVALUACION]] <br> De acuerdo con la tabla anterior 
podemos concluir que \\(\\beta_0\\) es significativamente [[cloze_sigB0]]  cero y que \\(\\beta_1\\)
es significativamente [[cloze_sigB1]] cero, adicionalmente podemos afirmar que la variable \\(LH\\)
[[close_conclu]] explicar a la variable <code>peso</code> </p>'  

In [28]:
coe=res[['coefficients']]
coe


,Estimate,Std. Error,t value,Pr(>|t|)
(Intercept),-507.2956,28.353660,-17.89171,2.387997e-13
LongHorquilla,37.5283,1.489697,25.19191,4.625199e-16


In [29]:
quan=quantile(df$LongHorquilla,probs = c(0.1,0.35,0.55,0.67,0.85))
quan
dfnw=data.frame(LongHorquilla=quan)
confl=0.95
prd=predict(mod2,newdata=dfnw,interval="confidence",level=confl)
prd
respuestas_dinamicas3 <- list(
    conf = confl*100, 
  p1     = quan[1],
  p2     = quan[2],
  p3     = quan[3],
  p4     = quan[4],
  p5     = quan[5],
  cloze_fit1= cloze_num(prd[1,1]),
  cloze_fit2= cloze_num(prd[2,1]),
  cloze_fit3= cloze_num(prd[3,1]),
  cloze_fit4= cloze_num(prd[4,1]),
  cloze_fit5= cloze_num(prd[5,1]),
  cloze_lwr1= cloze_num(prd[1,2]),
  cloze_lwr2= cloze_num(prd[2,2]),
  cloze_lwr3= cloze_num(prd[3,2]),
  cloze_lwr4= cloze_num(prd[4,2]),
  cloze_lwr5= cloze_num(prd[5,2]),
  cloze_upr1= cloze_num(prd[1,3]),
  cloze_upr2= cloze_num(prd[2,3]),
  cloze_upr3= cloze_num(prd[3,3]),
  cloze_upr4= cloze_num(prd[4,3]),
  cloze_upr5= cloze_num(prd[5,3])
) 

plantilla_tabla3<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0lax">LH</th>
    <th class="tg-0lax">fit</th>
    <th class="tg-0lax">lwr</th>
    <th class="tg-0lax">upr</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0lax">[[p1]]</td>
    <td class="tg-0lax">[[cloze_fit1]]</td>
    <td class="tg-0lax">[[cloze_lwr1]]</td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr1]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr2]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr3]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr4]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr5]]</span></td>
  </tr>
</tbody></table>'  


plantilla_enunciado3<-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>.  Se quiere  ajustar un modelo que permita explicar el <code>peso</code> de los animales en función de las longitudes. 
Ajuste el modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\)  
<br> Complete la siguiente tabla que contiene las estimaciones y límites
de confianza al [[conf]]% para el peso promedio de las poblaciones de cachamas que tienen LH = [[p1]], [[p2]], [[p3]], [[p4]], [[p5]] <br>[[TABLA_EVALUACION]] <br>
</p>'

10%  35%  55%  67%  85% 
17.0 18.0 18.5 18.5 20.0

,fit,lwr,upr
10%,130.6855,120.7671,140.6039
35%,168.2138,159.7293,176.6984
55%,186.9780,178.8487,195.1072
67%,186.9780,178.8487,195.1072
85%,243.2704,234.4646,252.0763


In [30]:
prd=predict(mod2,newdata=dfnw,interval="prediction",level=confl)
prd
respuestas_dinamicas4 <- list(
  conf = confl*100,
  p1     = quan[1],
  p2     = quan[2],
  p3     = quan[3],
  p4     = quan[4],
  p5     = quan[5],
  cloze_fit1= cloze_num(prd[1,1]),
  cloze_fit2= cloze_num(prd[2,1]),
  cloze_fit3= cloze_num(prd[3,1]),
  cloze_fit4= cloze_num(prd[4,1]),
  cloze_fit5= cloze_num(prd[5,1]),
  cloze_lwr1= cloze_num(prd[1,2]),
  cloze_lwr2= cloze_num(prd[2,2]),
  cloze_lwr3= cloze_num(prd[3,2]),
  cloze_lwr4= cloze_num(prd[4,2]),
  cloze_lwr5= cloze_num(prd[5,2]),
  cloze_upr1= cloze_num(prd[1,3]),
  cloze_upr2= cloze_num(prd[2,3]),
  cloze_upr3= cloze_num(prd[3,3]),
  cloze_upr4= cloze_num(prd[4,3]),
  cloze_upr5= cloze_num(prd[5,3])
) 

plantilla_tabla4<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0lax">LH</th>
    <th class="tg-0lax">fit</th>
    <th class="tg-0lax">lwr</th>
    <th class="tg-0lax">upr</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0lax">[[p1]]</td>
    <td class="tg-0lax">[[cloze_fit1]]</td>
    <td class="tg-0lax">[[cloze_lwr1]]</td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr1]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr2]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr3]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr4]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr5]]</span></td>
  </tr>
</tbody></table>'  


plantilla_enunciado4<-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>. Se quiere  ajustar un modelo que permita explicar el <code>peso</code> de los animales en función de las longitudes. 
Ajuste el modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\)   
<br> Complete la siguiente tabla que contiene las estimaciones y límites
de confianza al [[conf]]% para  una cachama que tiene LH = [[p1]], [[p2]], [[p3]], [[p4]], [[p5]] <br>[[TABLA_EVALUACION]] <br>
</p>'

,fit,lwr,upr
10%,130.6855,92.47411,168.8970
35%,168.2138,130.34927,206.0784
55%,186.9780,149.19144,224.7645
67%,186.9780,149.19144,224.7645
85%,243.2704,205.33259,281.2083


In [31]:
lev=0.99
prd=predict(mod2,newdata=dfnw,interval="confidence", level=lev)
prd
levp=lev*100
respuestas_dinamicas5 <- list(
    conf = levp,
  p1     = quan[1],
  p2     = quan[2],
  p3     = quan[3],
  p4     = quan[4],
  p5     = quan[5],
  cloze_fit1= cloze_num(prd[1,1]),
  cloze_fit2= cloze_num(prd[2,1]),
  cloze_fit3= cloze_num(prd[3,1]),
  cloze_fit4= cloze_num(prd[4,1]),
  cloze_fit5= cloze_num(prd[5,1]),
  cloze_lwr1= cloze_num(prd[1,2]),
  cloze_lwr2= cloze_num(prd[2,2]),
  cloze_lwr3= cloze_num(prd[3,2]),
  cloze_lwr4= cloze_num(prd[4,2]),
  cloze_lwr5= cloze_num(prd[5,2]),
  cloze_upr1= cloze_num(prd[1,3]),
  cloze_upr2= cloze_num(prd[2,3]),
  cloze_upr3= cloze_num(prd[3,3]),
  cloze_upr4= cloze_num(prd[4,3]),
  cloze_upr5= cloze_num(prd[5,3])
) 

plantilla_tabla5<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0lax">LH</th>
    <th class="tg-0lax">fit</th>
    <th class="tg-0lax">lwr</th>
    <th class="tg-0lax">upr</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0lax">[[p1]]</td>
    <td class="tg-0lax">[[cloze_fit1]]</td>
    <td class="tg-0lax">[[cloze_lwr1]]</td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr1]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr2]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr3]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr4]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr5]]</span></td>
  </tr>
</tbody></table>'  


plantilla_enunciado5<-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>.  Se quiere  ajustar un modelo que permita explicar el <code>peso</code> de los animales en función de las longitudes. 
Ajuste el modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\)  
<br> Complete la siguiente tabla que contiene las estimaciones y límites
de confianza al [[conf]]% para el peso promedio de las poblaciones de cachamas  que tienen LH = [[p1]], [[p2]], [[p3]], [[p4]], [[p5]] <br>[[TABLA_EVALUACION]] <br>
</p>'

,fit,lwr,upr
10%,130.6855,117.1282,144.2429
35%,168.2138,156.6164,179.8113
55%,186.9780,175.8662,198.0898
67%,186.9780,175.8662,198.0898
85%,243.2704,231.2338,255.3070


In [32]:
lev=0.99
prd=predict(mod2,newdata=dfnw,interval="prediction", level=lev)
prd
levp=lev*100
respuestas_dinamicas6 <- list(
    conf = levp,
  p1     = quan[1],
  p2     = quan[2],
  p3     = quan[3],
  p4     = quan[4],
  p5     = quan[5],
  cloze_fit1= cloze_num(prd[1,1]),
  cloze_fit2= cloze_num(prd[2,1]),
  cloze_fit3= cloze_num(prd[3,1]),
  cloze_fit4= cloze_num(prd[4,1]),
  cloze_fit5= cloze_num(prd[5,1]),
  cloze_lwr1= cloze_num(prd[1,2]),
  cloze_lwr2= cloze_num(prd[2,2]),
  cloze_lwr3= cloze_num(prd[3,2]),
  cloze_lwr4= cloze_num(prd[4,2]),
  cloze_lwr5= cloze_num(prd[5,2]),
  cloze_upr1= cloze_num(prd[1,3]),
  cloze_upr2= cloze_num(prd[2,3]),
  cloze_upr3= cloze_num(prd[3,3]),
  cloze_upr4= cloze_num(prd[4,3]),
  cloze_upr5= cloze_num(prd[5,3])
) 

plantilla_tabla6<- '<style type="text/css">
.tg  {border-collapse:collapse;border-color:#9ABAD9;border-spacing:0;}
.tg td{background-color:#EBF5FF;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#444;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#409cff;border-color:#9ABAD9;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0lax">LH</th>
    <th class="tg-0lax">fit</th>
    <th class="tg-0lax">lwr</th>
    <th class="tg-0lax">upr</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0lax">[[p1]]</td>
    <td class="tg-0lax">[[cloze_fit1]]</td>
    <td class="tg-0lax">[[cloze_lwr1]]</td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr1]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit2]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr2]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit3]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr3]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit4]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr4]]</span></td>
  </tr>
  <tr>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[p5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_fit5]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_lwr1]]</span></td>
    <td class="tg-0lax"><span style="font-weight:400;font-style:normal;text-decoration:none">[[cloze_upr5]]</span></td>
  </tr>
</tbody></table>'  


plantilla_enunciado6<-'
<p>El archivo de datos <code>DATOS CACHAMAS.xlsx</code> contiene datos de biometría de cachama blanca,   
<code>peso</code> en gramos y las longitudes de horquilla <code>LH</code>, estándar <code>LE</code> y total <code>LT</code> 
en gramos. Además, la columna <code>estanque</code> clasifica a los animales en dos estanques, <code>R02</code> y <code>R03</code>.
<span style="color: red; font-weight: bold;"> Para este ejercicio se deben usar solo los datos del estanque</span> <code>R03</code>.  Se quiere  ajustar un modelo que permita explicar el <code>peso</code> de los animales en función de las longitudes. 
Ajuste el modelo \\(E(P_i)=\\beta_0+\\beta_1LH_i\\)   
<br> Complete la siguiente tabla que contiene las estimaciones y límites
de confianza al [[conf]]% para el peso promedio de las poblaciones de cachamas que tienen una longitud de horquilla
LH = [[p1]], [[p2]], [[p3]], [[p4]], [[p5]] <br>[[TABLA_EVALUACION]] <br>
</p>'

,fit,lwr,upr
10%,130.6855,78.4547,182.9164
35%,168.2138,116.4571,219.9705
55%,186.9780,135.3279,238.6281
67%,186.9780,135.3279,238.6281
85%,243.2704,191.4136,295.1273


In [33]:

# 1. Creamos o abrimos el archivo XML de Moodle bajo una categoría
banco_moodle <- iniciar_quiz_xml(nombre_categoria = "Laboratorio/SeleccionModelosCachamas2")

# 2. Inyectamos la pregunta mandando las plantillas y los cálculos dinámicos de R
crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "Todas_Vs_LH",
  plantilla_texto  = plantilla_enunciado,
  plantilla_tabla  = plantilla_tabla,
  lista_respuestas = respuestas_dinamicas
)

crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "hipotesis_betas_peso",
  plantilla_texto  = plantilla_enunciado2,
  plantilla_tabla  = plantilla_tabla2,
  lista_respuestas = respuestas_dinamicas2
)

crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "prediccion1Cach",
  plantilla_texto  = plantilla_enunciado3,
  plantilla_tabla  = plantilla_tabla3,
  lista_respuestas = respuestas_dinamicas3
)

crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "prediccion2Cach",
  plantilla_texto  = plantilla_enunciado4,
  plantilla_tabla  = plantilla_tabla4,
  lista_respuestas = respuestas_dinamicas4
)

crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "prediccion3Cach",
  plantilla_texto  = plantilla_enunciado5,
  plantilla_tabla  = plantilla_tabla5,
  lista_respuestas = respuestas_dinamicas5
)

crear_pregunta_cloze(
  moodle_quiz      = banco_moodle,
  nombre_pregunta  = "prediccion4Cach",
  plantilla_texto  = plantilla_enunciado6,
  plantilla_tabla  = plantilla_tabla6,
  lista_respuestas = respuestas_dinamicas6
)


<idnumber/> 

<idnumber/> 

<idnumber/> 

<idnumber/> 

<idnumber/> 

<idnumber/> 

In [34]:
# 3. Guardamos el archivo xml listo para Moodle
saveXML(banco_moodle$doc, file = "Banco_Seleccion_ModeloCachamas2.xml") 

[1] "Banco_Seleccion_ModeloCachamas2.xml"

ERROR: Error in glue_data(lista_respuestas, plantilla_tabla): Expecting '}'


In [ ]:
# Definimos el molde de la tabla descriptiva básica
plantilla_tabla_descriptiva <- '
<table border="1" style="border-collapse:collapse; padding:5px; text-align:center;">
  <thead>
    <tr style="background-color:#f38630; color:white;">
      <th>Métrica</th>
      <th>Valor Requerido</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mínimo</td><td>{cloze_min}</td></tr>
    <tr><td>Mediana</td><td>{cloze_mediana}</td></tr>
    <tr><td>Media Muestral</td><td>{cloze_media}</td></tr>
  </tbody>
</table>'

# Definimos el molde para el enunciado del ítem
plantilla_enunciado <- "
<p>El archivo de datos contiene variables de control biométrico. Analice la columna y responda:</p>
<p>¿Cuántos registros faltantes (NA) posee esta columna?: {cloze_nas}</p>
<p>¿Cuál es la función de <code>tidyverse</code> adecuada para computar resúmenes estadísticos?: {cloze_comando}</p>
<p>Complete la tabla estadística de resumen:</p>
{TABLA_EVALUACION}
"